<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/DNA_Sequence_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  DNA Sequence Classifier

# Project 4  — Bioinformatics

# Is notebook mein hum DNA sequences ko classify karain gay ke woh Promoter region hain ya Non-promoter k-mer frequency features aur Machine Learning use kar ke.

# **Pipeline:**
# 1. Sequence data generate/load karna
# 2. Feature extraction (k-mer frequencies, GC content, sequence composition)
# 3. Interactive Plotly visualizations
# 4. Model training (multiple classifiers compare)
# 5. Evaluation (accuracy, ROC-AUC, confusion matrix, top predictive k-mers)
# 6.Runtime cell  apni khud ki DNA sequence daal kar direct predict karein

>


## 1. Setup & Imports

In [1]:
# !pip install -q plotly scikit-learn pandas numpy ipywidgets

import numpy as np
import pandas as pd
import re
from itertools import product

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, roc_curve

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


## 2. Generate DNA Sequence Dataset


In [2]:
def random_seq(length, rng):
    return "".join(rng.choice(list("ACGT"), size=length))

def generate_dataset(n_per_class=300, seq_len=60, seed=42):
    rng = np.random.default_rng(seed)
    sequences, labels = [], []

    motifs = ["TATAAA", "TATATA", "TATAAT"]  # TATA-box-like motifs (promoter signal)

    for _ in range(n_per_class):
        # Promoter: random sequence with a TATA-box-like motif inserted
        seq = list(random_seq(seq_len, rng))
        motif = rng.choice(motifs)
        insert_pos = rng.integers(5, seq_len - len(motif) - 5)
        seq[insert_pos:insert_pos + len(motif)] = list(motif)
        sequences.append("".join(seq))
        labels.append(1)  # Promoter

    for _ in range(n_per_class):
        # Non-promoter: pure random sequence (motif absent, by construction rare)
        seq = random_seq(seq_len, rng)
        sequences.append(seq)
        labels.append(0)  # Non-promoter

    return sequences, labels

sequences, labels = generate_dataset(n_per_class=300, seq_len=60)
seq_df = pd.DataFrame({"sequence": sequences, "label": labels})
seq_df["class"] = seq_df["label"].map({1: "Promoter", 0: "Non-promoter"})

print(f"Total sequences: {len(seq_df)}")
seq_df.head()


Total sequences: 600


,sequence,label,class
0,ATGCCTAGAAGTGTGTGATCGCATTGCTGCCAAGTATTATAATCAT...,1,Promoter
1,CTCCTCACTACAGCCAGGTCATGGATATATACTCAGGATATATTTG...,1,Promoter
2,TGATGGGGAGTCGACCTACCTTAATATATACGAGGTTGCCCTCACA...,1,Promoter
3,ACACGGGCTACACTCTCGCCTTCTATAATCAACTACGAGCTGGACT...,1,Promoter
4,TAACACGAGTATAAATTGCCGGCAATCCCTAACCGCCTAGGCTCTC...,1,Promoter


## 3. Feature Extraction — k-mers & Composition

In [3]:
def gc_content(seq):
    return (seq.count("G") + seq.count("C")) / len(seq)

def kmer_frequencies(seq, k=3):
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    total = len(kmers)
    freq = {}
    for km in kmers:
        freq[km] = freq.get(km, 0) + 1
    return {km: c / total for km, c in freq.items()}

K = 3
all_kmers = ["".join(p) for p in product("ACGT", repeat=K)]

def extract_features(seq_list, k=K):
    rows = []
    for seq in seq_list:
        freqs = kmer_frequencies(seq, k)
        row = {km: freqs.get(km, 0.0) for km in all_kmers}
        row["gc_content"] = gc_content(seq)
        row["length"] = len(seq)
        rows.append(row)
    return pd.DataFrame(rows)

feature_df = extract_features(sequences)
feature_df["label"] = labels
feature_df["class"] = seq_df["class"]

print(f"Feature matrix shape: {feature_df.shape}  ({len(all_kmers)} k-mers + gc_content + length)")
feature_df.head()


Feature matrix shape: (600, 68)  (64 k-mers + gc_content + length)


,AAA,AAC,AAG,AAT,ACA,ACC,ACG,ACT,AGA,AGC,...,TGG,TGT,TTA,TTC,TTG,TTT,gc_content,length,label,class
0,0.000000,0.000000,0.034483,0.017241,0.000000,0.017241,0.000000,0.000000,0.034483,0.000000,...,0.000000,0.051724,0.034483,0.000000,0.017241,0.000000,0.433333,60,1,Promoter
1,0.034483,0.017241,0.000000,0.000000,0.017241,0.000000,0.017241,0.034483,0.000000,0.017241,...,0.017241,0.000000,0.000000,0.000000,0.017241,0.017241,0.466667,60,1,Promoter
2,0.017241,0.000000,0.000000,0.034483,0.017241,0.034483,0.034483,0.000000,0.000000,0.000000,...,0.034483,0.017241,0.017241,0.000000,0.017241,0.000000,0.483333,60,1,Promoter
3,0.000000,0.017241,0.000000,0.017241,0.034483,0.000000,0.034483,0.051724,0.017241,0.017241,...,0.017241,0.000000,0.000000,0.017241,0.000000,0.000000,0.533333,60,1,Promoter
4,0.017241,0.034483,0.000000,0.034483,0.017241,0.017241,0.034483,0.017241,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.017241,0.000000,0.516667,60,1,Promoter


## 4. Exploratory Visualizations

In [4]:
fig = px.histogram(
    feature_df, x="gc_content", color="class", barmode="overlay", nbins=30,
    title="GC Content Distribution — Promoter vs Non-promoter",
    template="plotly_white",
    color_discrete_map={"Promoter": "#E63946", "Non-promoter": "#2E86AB"},
    opacity=0.65
)
fig.update_layout(height=500)
fig.show()

class_counts = seq_df["class"].value_counts()
fig2 = px.pie(names=class_counts.index, values=class_counts.values, hole=0.45,
              title="Class Balance", color=class_counts.index,
              color_discrete_map={"Promoter": "#E63946", "Non-promoter": "#2E86AB"})
fig2.update_layout(height=450)
fig2.show()


In [5]:
# Average k-mer frequency comparison between classes
mean_by_class = feature_df.groupby("class")[all_kmers].mean().T
mean_by_class["diff"] = mean_by_class["Promoter"] - mean_by_class["Non-promoter"]
top_diff = mean_by_class.reindex(mean_by_class["diff"].abs().sort_values(ascending=False).index).head(15)

fig = go.Figure()
fig.add_trace(go.Bar(x=top_diff.index, y=top_diff["Promoter"], name="Promoter", marker_color="#E63946"))
fig.add_trace(go.Bar(x=top_diff.index, y=top_diff["Non-promoter"], name="Non-promoter", marker_color="#2E86AB"))
fig.update_layout(
    title="Top 15 Most Discriminative 3-mers (Average Frequency)",
    xaxis_title="3-mer", yaxis_title="Average Frequency",
    barmode="group", template="plotly_white", height=500
)
fig.show()


In [6]:
# Heatmap of k-mer frequencies across a sample of sequences
sample_idx = np.concatenate([
    feature_df[feature_df["label"] == 1].sample(15, random_state=1).index,
    feature_df[feature_df["label"] == 0].sample(15, random_state=1).index
])
heat_data = feature_df.loc[sample_idx, all_kmers]
heat_labels = feature_df.loc[sample_idx, "class"]

fig = px.imshow(
    heat_data.values, x=all_kmers, y=[f"{c} #{i}" for i, c in enumerate(heat_labels)],
    color_continuous_scale="Viridis", aspect="auto",
    title="k-mer Frequency Heatmap (30 sample sequences)",
    labels=dict(color="Frequency")
)
fig.update_layout(height=650, xaxis_tickangle=-90)
fig.show()


## 5. Train & Compare Classifiers

In [7]:
feature_cols = all_kmers + ["gc_content", "length"]
X = feature_df[feature_cols]
y = feature_df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

dna_scaler = StandardScaler()
X_train_scaled = dna_scaler.fit_transform(X_train)
X_test_scaled = dna_scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "SVM (RBF)": SVC(probability=True, random_state=42),
}

results = []
trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    preds = model.predict(X_test_scaled)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs)
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
results_df


,Model,Accuracy,ROC-AUC
0,Random Forest,0.800000,0.868533
1,Logistic Regression,0.753333,0.836444
2,SVM (RBF),0.760000,0.830756


In [8]:
fig = px.bar(
    results_df.melt(id_vars="Model", value_vars=["Accuracy", "ROC-AUC"]),
    x="Model", y="value", color="variable", barmode="group",
    title="Model Comparison", template="plotly_white",
    labels={"value": "Score", "variable": "Metric"}
)
fig.update_layout(height=450, yaxis_range=[0.5, 1.05])
fig.show()

best_model_name = results_df.iloc[0]["Model"]
dna_clf = trained_models[best_model_name]
print(f"Best model: {best_model_name}")


Best model: Random Forest


## 6. Detailed Evaluation

In [9]:
preds = dna_clf.predict(X_test_scaled)
probs = dna_clf.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, preds, target_names=["Non-promoter", "Promoter"]))

cm = confusion_matrix(y_test, preds)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                 x=["Non-promoter", "Promoter"], y=["Non-promoter", "Promoter"],
                 labels=dict(x="Predicted", y="Actual", color="Count"),
                 title=f"Confusion Matrix — {best_model_name}")
fig.update_layout(height=450, width=500)
fig.show()

fpr, tpr, _ = roc_curve(y_test, probs)
auc = roc_auc_score(y_test, probs)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC (AUC={auc:.3f})', line=dict(color="#E63946", width=3)))
fig2.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random', line=dict(dash='dash', color='gray')))
fig2.update_layout(title="ROC Curve", xaxis_title="False Positive Rate", yaxis_title="True Positive Rate",
                    template="plotly_white", height=500)
fig2.show()


              precision    recall  f1-score   support

Non-promoter       0.83      0.76      0.79        75
    Promoter       0.78      0.84      0.81        75

    accuracy                           0.80       150
   macro avg       0.80      0.80      0.80       150
weighted avg       0.80      0.80      0.80       150



In [10]:
if hasattr(dna_clf, "feature_importances_"):
    importances = pd.Series(dna_clf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)
    fig = px.bar(importances, orientation='h', title=f"Top 15 Feature Importances — {best_model_name}",
                 labels={"value": "Importance", "index": "Feature"}, template="plotly_white",
                 color=importances.values, color_continuous_scale="Viridis")
    fig.update_layout(height=500, showlegend=False, yaxis={'categoryorder': 'total ascending'})
    fig.show()
else:
    print(f"{best_model_name} feature_importances_ nahi deta — coefficient use kar sakte hain (Logistic Regression ke liye).")


## 7.  Runtime Prediction — Apni DNA Sequence Direct Input Karein




In [11]:
seq_input_box = widgets.Textarea(
    value="GCGATCGTATAAACGTAGCTAGCTGATCGATCGATCGTAGCTAGCATCGATCGTAGCTA",
    description="DNA Sequence:",
    placeholder="Sirf A, C, G, T letters likhein...",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='650px', height='80px')
)

predict_btn = widgets.Button(description=" Sequence Predict Karein", button_style='success',
                              layout=widgets.Layout(width='260px', height='42px'))
out = widgets.Output()

def render_result(label, proba, seq_clean, gc):
    color = "#E63946" if label == "Promoter" else "#2E86AB"
    emoji = "🔴" if label == "Promoter" else "🔵"
    conf = proba[1]*100 if label == "Promoter" else proba[0]*100
    html = f"""
    <div style="border:2px solid {color}; border-radius:12px; padding:18px; margin-top:10px; font-family:sans-serif; background:#fafafa;">
        <div style="font-size:22px; font-weight:700; color:{color};">{emoji} Prediction: {label}</div>
        <div style="font-size:13px; color:#555; margin-top:4px;">Length: {len(seq_clean)} bp &nbsp;|&nbsp; GC content: {gc*100:.1f}%</div>
        <div style="font-size:15px; margin-top:8px;">Confidence: <b>{conf:.2f}%</b></div>
        <div style="margin-top:10px; height:14px; width:100%; background:#e0e0e0; border-radius:7px; overflow:hidden;">
            <div style="height:100%; width:{proba[1]*100:.1f}%; background:linear-gradient(90deg,#2E86AB,#E63946);"></div>
        </div>
        <div style="display:flex; justify-content:space-between; font-size:12px; color:#555; margin-top:4px;">
            <span>P(Non-promoter) = {proba[0]*100:.2f}%</span><span>P(Promoter) = {proba[1]*100:.2f}%</span>
        </div>
    </div>
    """
    display(HTML(html))

def on_predict(b):
    with out:
        clear_output()
        raw_seq = seq_input_box.value.strip().upper()
        seq_clean = re.sub(r"[^ACGT]", "", raw_seq)

        if len(seq_clean) < K:
            print(f"⚠️ Sequence bahut chhoti hai (kam az kam {K} bases chahiye).")
            return

        freqs = kmer_frequencies(seq_clean, K)
        row = {km: freqs.get(km, 0.0) for km in all_kmers}
        row["gc_content"] = gc_content(seq_clean)
        row["length"] = len(seq_clean)
        row_df = pd.DataFrame([row])[feature_cols]

        row_scaled = dna_scaler.transform(row_df)
        pred = dna_clf.predict(row_scaled)[0]
        proba = dna_clf.predict_proba(row_scaled)[0]
        label = "Promoter" if pred == 1 else "Non-promoter"
        render_result(label, proba, seq_clean, row["gc_content"])

predict_btn.on_click(on_predict)

display(seq_input_box)
display(predict_btn)
display(out)


Textarea(value='GCGATCGTATAAACGTAGCTAGCTGATCGATCGATCGTAGCTAGCATCGATCGTAGCTA', description='DNA Sequence:', lay…

Button(button_style='success', description=' Sequence Predict Karein', layout=Layout(height='42px', width='260…

Output()